# 06 Downstream analysis: using the codes and embeddings

Training attaches four things to your AnnData: the discrete `cell_codebook_idx` and
`neighborhood_codebook_idx`, and the continuous `X_cell_embedding` /
`X_neighborhood_embedding`. This notebook shows the three things you most often do
with them, all with standard scanpy / pandas:

1. a UMAP of the continuous cell embedding, colored by cell code;
2. the cell-code x niche composition, which is how the two codebooks connect (which
   cell states populate which tissue niches);
3. per-sample code usage.

Everything here is offline and deterministic.

In [ ]:
import os
import numpy as np, pandas as pd
import scanpy as sc
import nicheverse as nv
from nicheverse import ModelConfig, TrainConfig
DATA = os.path.join('..', 'examples', 'data')

adata = nv.read_spatial(os.path.join(DATA, 'merfish_retina.h5ad'), sample_col='sample_id')
mc = ModelConfig(input_dim=adata.n_vars, gene_names=tuple(adata.var_names.astype(str)),
                 encoder_type='mlp_deep', cell_num_embeddings=256, neighborhood_num_embeddings=32)
tc = TrainConfig(num_epochs=15, batch_size=2048, spatial_graph='knn_radius', radius=50.0,
                 k_neighbors=20, save_best=False, seed=9)
model, adata = nv.train_model(adata, 'runs/downstream_demo', model_config=mc, train_config=tc,
                              sample_col='sample_id')
print(nv.anndata_keys())
print('embedding:', adata.obsm['X_cell_embedding'].shape)

## 1. A 2D map of the cell embedding

The continuous embedding is a drop-in `use_rep` for the standard scanpy neighbors +
UMAP. We try UMAP and fall back to a 2D PCA if `umap-learn` is unavailable, so the
cell runs in any environment. Coloring by `cell_codebook_idx` shows that the discrete
codes carve the embedding into coherent regions.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt   # after importing nicheverse (which selects Agg), re-assert inline
emb = np.asarray(adata.obsm['X_cell_embedding'])
try:
    sc.pp.neighbors(adata, use_rep='X_cell_embedding', n_neighbors=15)
    sc.tl.umap(adata)
    xy = adata.obsm['X_umap']; method = 'UMAP'
except Exception as e:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2, random_state=0).fit_transform(emb); method = 'PCA'
    print('umap-learn unavailable, using 2D PCA instead:', type(e).__name__)
codes = adata.obs['cell_codebook_idx'].to_numpy()
fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.scatter(xy[:, 0], xy[:, 1], c=codes, s=2, cmap='tab20', linewidths=0)
ax.set_xlabel(f'{method}1'); ax.set_ylabel(f'{method}2')
ax.set_title(f'cell embedding ({method}), colored by cell code')
ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); plt.show()
print('cells:', adata.n_obs, ' distinct cell codes:', len(np.unique(codes)))

## 2. Cell-code x niche composition

This is the join between the two codebooks: for each niche, what mixture of cell
codes lives there. A block structure means niches are stereotyped (each recurs with
a characteristic cell-code composition), which is what makes the niche codebook a
map of tissue organization.

In [ ]:
comp = pd.crosstab(adata.obs['cell_codebook_idx'], adata.obs['neighborhood_codebook_idx'])
comp = comp.div(comp.sum(0), axis=1)  # column-normalize: composition within each niche
top_codes = comp.sum(1).sort_values(ascending=False).head(25).index
M = comp.loc[top_codes]
fig, ax = plt.subplots(figsize=(6.2, 5.0))
im = ax.imshow(M.values, aspect='auto', cmap='magma')
ax.set_xlabel('niche code'); ax.set_ylabel('cell code (top 25 by usage)')
ax.set_title('cell-code composition within each niche')
ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(M.columns, fontsize=6, rotation=90)
ax.set_yticks(range(M.shape[0])); ax.set_yticklabels(M.index, fontsize=6)
fig.colorbar(im, ax=ax, fraction=0.046, label='fraction of niche')
fig.tight_layout(); plt.show()

## 3. Per-sample code usage

A quick check that the codebook is shared across samples rather than one code per
sample. Each row is a sample; each column a cell code; the value is the fraction of
that sample's cells in the code.

In [ ]:
usage = pd.crosstab(adata.obs['sample_id'], adata.obs['cell_codebook_idx'], normalize='index')
print('samples:', usage.shape[0], ' codes:', usage.shape[1])
usage.iloc[:, :12].round(3)

## Takeaways

- The continuous embeddings plug straight into the scanpy neighbors / UMAP /
  clustering stack; the discrete codes are a ready-made categorical for coloring and
  grouping.
- The cell-code x niche crosstab is the bridge between the two codebooks and the
  starting point for compositional and enrichment analyses (for example, testing
  whether a cell state is enriched in a particular niche across conditions).
- Because the codebook is shared, per-sample usage is comparable across samples and
  studies, which is what makes the codes a transferable annotation.